# --- Begin: Project Overview ---

## Breast Cancer Treatment Prediction

**Objective:** The project aims to develop a model that can predict the best treatment strategy (endocrine therapy, chemotherapy, both, or neither) for breast cancer patients based on their gene expression data.

**Data:** The code uses two primary datasets:
*   **Genomic Data:** Gene expression data from RNA sequencing.
*   **Clinical Data:** Patient information, including age, tumor size, lymph node status, ER/PR/HER2 status, Ki67 status, Nottingham grade (NHG), PAM50 subtype, survival information (overall survival days and event), and treatment history (endocrine therapy, chemotherapy).

**Methodology:**
*   **Data Preprocessing:** Cleans and merges genomic and clinical data.
*   **Feature Selection/Extraction:** Employs two techniques:
    *   LASSO regression for feature selection (identifying relevant genes).
    *   Autoencoders for dimensionality reduction and feature extraction (encoding gene expression data into a lower-dimensional representation).
*   **Clinical Data Prediction:** Trains multi-output classification models (Logistic Regression) to predict clinical variables from:
    *   Encoded genomic data.
    *   Feature-selected genomic data.
*   **Survival Prediction:** Develops a survival model (Cox Proportional Hazards model with LASSO regularization) to predict patient survival based on clinical data.
*   **Treatment Recommendation:** Combines the clinical and survival models to recommend the optimal treatment strategy for new patients.


In [ ]:
# --- Beginr: Importing Libraries ---
# Essential libraries for data manipulation, machine learning, and survival analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, classification_report, matthews_corrcoef, mean_squared_error
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
import umap
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

In [ ]:
pwd

## Section 1: Data Loading and Cleaning

In [ ]:
# Load datasets and perform initial cleaning and merging
# Define file paths
expression_matrix = "Data/GSE96058_gene_expression_3273_samples_and_136_replicates_transformed.csv"
clinical_data_1 = "Data/GSE96058-GPL11154_series_matrix.txt.csv"
clinical_data_2 = "Data/GSE96058-GPL18573_series_matrix.txt.csv"

# Load dataframes
expression_matrix_df = pd.read_csv(expression_matrix, index_col=0).T
clinical_data_df_1 = pd.read_csv(clinical_data_1, index_col=0)
clinical_data_df_2 = pd.read_csv(clinical_data_2, index_col=0)

# Concatenate clinical dataframes
clinical_data_df = pd.concat([clinical_data_df_1, clinical_data_df_2], axis=0)
del clinical_data_df_1, clinical_data_df_2




Send to categorical data

In [ ]:


clinical_data_df_cat = clinical_data_df.copy()
# Encode categorical variables
for col in clinical_data_df_cat.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    clinical_data_df_cat[col] = le.fit_transform(clinical_data_df_cat[col].astype(str))

Divide X as the genes and Y as the clinical variables

In [ ]:
# Merge genomic and clinical data
XY = expression_matrix_df.merge(clinical_data_df_cat, left_index=True, right_index=True)
# replace NAN with closest neighbor
XY.fillna(method='ffill', inplace=True)
# count the number of missing values
missing_values_count = XY.isnull().sum()
# Display missing values count
print(f"Missing values: {sum(missing_values_count)}")
XY.dropna(inplace=True)


In [ ]:

# Split into features (X) and target (y)
X = XY[expression_matrix_df.columns]
y = XY[clinical_data_df_cat.columns]
columns_interest=['er status', 'pgr status', 'her2 status',
       'ki67 status', 'nhg', 'er prediction mgc', 'pgr prediction mgc',
       'her2 prediction mgc', 'ki67 prediction mgc', 'nhg prediction mgc',
       'er prediction sgc', 'pgr prediction sgc', 'her2 prediction sgc',
       'ki67 prediction sgc', 'pam50 subtype', 'overall survival days',
       'overall survival event', 'endocrine treated', 'chemo treated']
# Filter columns of interest
y = y[columns_interest]


In [ ]:

print(f"Number of samples after merging and cleaning: {len(XY)}")
print("Data loading and merging complete.")

## Section 2: Feature Selection and Extraction

To proceed with the prediction, we need to see which genes better describe the outcome of a patient. We know that PAM 50 is based on 50 genes to be calculated. And that PAM 50 is a good representation of the markers of Cancer (er, etc). 
But here we are falling on an assumtion that we don't know is true. 
Therefore, we can try to find a subset of genes or combinatory (autoencoder) that can predict the clinical values.

In [ ]:
# Get the clinical features of interest for the clinical data
# 1. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y[['er status', 'pgr status', 'her2 status','ki67 status', 
                                                       'nhg', 'pam50 subtype']], 
                                                      test_size=0.2, random_state=42)

# 2. Scale Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### LASSO

LASSO regression allows a binary selection of the genes. 

In [ ]:
# Lasso Feature Selection
lasso = Lasso(alpha=0.1) 
lasso.fit(X_train_scaled, y_train)

# Select Features
selector = SelectFromModel(lasso, prefit=True)
X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)

# Get the selected feature names (genes)
selected_gene_indices = selector.get_support()
selected_gene_names = X.columns[selected_gene_indices]

print(f"Number of selected genes: {len(selected_gene_names)}")
print("LASSO feature selection complete.")

In [ ]:
y_train.columns

### Autoencoder

An autoencoder will allow us to reduce the dimentionallity but still converving all the genes information

In [ ]:
# Autoencoder
input_dim = X_train_scaled.shape[1]  # Number of genes
encoding_dim = 128  # Bottleneck dimension

# Input Layer
input_layer = Input(shape=(input_dim,))

# Encoder
encoder = Dense(512, activation='relu')(input_layer)
encoder = Dense(256, activation='relu')(encoder)
encoder = Dense(encoding_dim, activation='relu')(encoder)  # Bottleneck layer

# Decoder
decoder = Dense(256, activation='relu')(encoder)
decoder = Dense(512, activation='relu')(decoder)
decoder = Dense(input_dim, activation='linear')(decoder)  # Output layer

# Autoencoder Model
autoencoder = Model(inputs=input_layer, outputs=decoder)

# Model Training
autoencoder.compile(optimizer='adam', loss='mse')

# Train the autoencoder
autoencoder.fit(X_train_scaled, X_train_scaled,
                epochs=50,  # Adjust as needed
                batch_size=32, # Adjust as needed
                shuffle=True,
                validation_data=(X_test_scaled, X_test_scaled))

# Feature Extraction
# Create the encoder model (from input to bottleneck layer)
encoder_model = Model(inputs=input_layer, outputs=encoder)

# Encode the data
X_train_encoded = encoder_model.predict(X_train_scaled)
X_test_encoded = encoder_model.predict(X_test_scaled)

print("Autoencoder training and feature extraction complete.")

### Saving the models and data

In [255]:
import pickle
# Save the encoder model
with open('encoder_model.pkl', 'wb') as f:
    pickle.dump(encoder_model, f)
# Save the scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
# Save the selected gene names
with open('selected_genes.pkl', 'wb') as f:
    pickle.dump(selected_gene_names, f)
# Save the selected features
X_train_selected_df = pd.DataFrame(X_train_selected, columns=selected_gene_names)
X_test_selected_df = pd.DataFrame(X_test_selected, columns=selected_gene_names)
X_train_selected_df.to_csv('X_train_selected.csv', index=False)
X_test_selected_df.to_csv('X_test_selected.csv', index=False)
# Save the X_train_encoded and X_test_encoded
X_train_encoded_df = pd.DataFrame(X_train_encoded)
X_test_encoded_df = pd.DataFrame(X_test_encoded)
X_train_encoded_df.to_csv('X_train_encoded.csv', index=False)
X_test_encoded_df.to_csv('X_test_encoded.csv', index=False)
# save the clinical data
y_train_df = pd.DataFrame(y_train)
y_test_df = pd.DataFrame(y_test)
y_train_df.to_csv('y_train.csv', index=False)
y_test_df.to_csv('y_test.csv', index=False)

# Save selected genes in txt format
with open('selected_genes.txt', 'w') as f:
    for gene in selected_gene_names:
        f.write(f"{gene}\n")
print("Models and data saved.")

Models and data saved.


## Section 3: Predicting the clinical data 

### Prepare data

Among the Clinical parameters, ['er status', 'pgr status', 'ki67 status', 'her2 status', 'pam50 subtype', 'nhg']  are the only that we need to predict. 

In [ ]:
# Prepare clinical data
clinical_vars = ['er status', 'pgr status', 'ki67 status', 'her2 status', 'pam50 subtype', 'nhg']  # Example clinical vars
y_train_subset = y_train[clinical_vars]
y_test_subset = y_test[clinical_vars]

# Ensure y_train and y_test are DataFrames
if not isinstance(y_train, pd.DataFrame):
    y_train = pd.DataFrame(y_train)
if not isinstance(y_test, pd.DataFrame):
    y_test = pd.DataFrame(y_test)
    
# 3. Preprocess Each Column

for col in y_train_subset.columns:
    print(f"Column preprocessing: {col}")

    # a. Integer type encoding
    if y_train_subset[col].dtype == 'object': #check data type
        print(f"Converting column {col} to numeric...")
        try:
            le = LabelEncoder()
            y_train_subset[col] = le.fit_transform(y_train_subset[col])
            y_test_subset[col] = le.transform(y_test_subset[col])
            print(f"Column {col} successfully converted to numeric.")

        except Exception as e:
            print(f"Error converting column {col}: {e}")
        print(f"Column {col} is not categorical or could not be converted.")

    # b. Numerical Column Processing (Scaling and Encoding)
    else:
        try:
            # Check if the column is already numeric
            y_train_subset[col] = pd.to_numeric(y_train_subset[col])
            y_test_subset[col] = pd.to_numeric(y_test_subset[col])
            print("Already numerical column")

        except ValueError as e:
            print(f"Error converting column {col} to numeric: {e}")

        # c. Print Summary
        num_unique = y_train_subset[col].nunique()
        print(f"Unique values in {col}: {num_unique}")


### Prepare model

#### Encoded

In [ ]:
# Define and train the model
multi_target_model_encoded = MultiOutputClassifier(LogisticRegression(solver='liblinear', max_iter=2000))

# Fit the model
multi_target_model_encoded.fit(X_train_encoded, y_train_subset)

# Make predictions
y_pred_encoded = multi_target_model_encoded.predict(X_test_encoded)

# Evaluate the model
y_pred_encoded_df = pd.DataFrame(y_pred_encoded, columns=y_test_subset.columns, index=y_test_subset.index)

for col in y_test_subset.columns:
    accuracy_encoded = accuracy_score(y_test_subset[col], y_pred_encoded_df[col])
    mcc = matthews_corrcoef(y_test_subset[col], y_pred_encoded_df[col])
    print(f'MCC for {col}: {mcc:.4f}')




#### Selected features

In [ ]:
# Feature selection as input
X_selected = X[selected_gene_names]  # Create a new DataFrame with only the selected genes

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X_selected, y[clinical_vars], test_size=0.2, random_state=42)

# 4. Scale the Gene Expression Data
scaler_selected = StandardScaler()
X_train_scaled = scaler_selected.fit_transform(X_train)
X_test_scaled = scaler_selected.transform(X_test)

# 6. Train Multi-Output Model
multi_output_model_selected = MultiOutputClassifier(LogisticRegression(max_iter=1000))

# 7. Fit the Model
multi_output_model_selected.fit(X_train_scaled, y_train)

# 8. Make Predictions
y_pred_selected = multi_output_model_selected.predict(X_test_scaled)

# 9. Evaluate Performance
# Print the performance of the output
for i, col in enumerate(y_test.columns):
    accuracy_selected = accuracy_score(y_test[col], y_pred_selected[:, i])
    mcc_selected = matthews_corrcoef(y_test[col], y_pred_selected[:, i])
    print(f'MCC for {col}: {mcc_selected:.4f}')
print("Clinical data prediction complete.")

### Saving the models

In [ ]:
# Saving the models
# Save the multi-output model
with open('multi_output_model_selected.pkl', 'wb') as f:
    pickle.dump(multi_output_model_selected, f)
with open('multi_output_model_encoded.pkl', 'wb') as f:
    pickle.dump(multi_target_model_encoded, f)
# Save the scaler for selected genes
with open('scaler_selected.pkl', 'wb') as f:
    pickle.dump(scaler_selected, f)
# Save the scaler for encoded features
with open('scaler_encoded.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## Section 4:  Develop Survival Model and Recommending Treatment

In [ ]:
def map_treatment(row):
    if row['endocrine treated'] == 1 and row['chemo treated'] == 0:
        return 1  # Only endocrine
    elif row['endocrine treated'] == 0 and row['chemo treated'] == 1:
        return 2  # Only chemo
    elif row['endocrine treated'] == 1 and row['chemo treated'] == 1:
        return 3  # Both
    else:
        return 0  # None

In [ ]:

clinical_data = clinical_data_df_cat.copy()
clinical_data.dropna(inplace=True)  # Drop rows with any NaN values
survival_vars =['age at diagnosis', 'tumor size', 'lymph node group',
       'lymph node status', 'er status', 'pgr status', 'her2 status',
       'ki67 status', 'nhg', 'pam50 subtype','endocrine treated', 'chemo treated']
x_clinical_data = clinical_data[survival_vars]
# convert binary variables to boolean: 'er status', 'pgr status', 'her2 status', 'ki67 status', 'endocrine treated', 'chemo treated'
for col in ['er status', 'pgr status', 'her2 status', 'ki67 status', 'endocrine treated', 'chemo treated']:
    x_clinical_data[col] = x_clinical_data[col].astype(bool)
# make sure all columns are categorical
for col in ['lymph node group','lymph node status', 'er status', 'pgr status', 'her2 status',
       'ki67 status', 'nhg', 'pam50 subtype','endocrine treated', 'chemo treated']:
    x_clinical_data[col] = x_clinical_data[col].astype('category')
    
y_clinical_data = clinical_data[['overall survival event','overall survival days']]
y_clinical_data['overall survival event'] = y_clinical_data['overall survival event'].astype(bool)

X_train_survival, X_test_survival, y_train_survival, y_test_survival = train_test_split(x_clinical_data,
                                                                                          y_clinical_data,
                                                                                          test_size=0.2, random_state=42)



In [ ]:
X_train_survival

In [ ]:
scaler = StandardScaler()
#X_train_survival = scaler.fit_transform(X_train_survival)
#X_test_survival = scaler.transform(X_test_survival)

# Create structured array for survival data
y_train_survival = y_train_survival.to_records(index=False)
# save scaler
with open('scaler_survival_clinical.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Cox Model with Lasso Regularization
estimator = CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.1,  fit_baseline_model=True) 
estimator.fit(X_train_survival, y_train_survival)


In [ ]:
pd.DataFrame(X_test_survival, columns= x_clinical_data.columns)

In [ ]:
# Predict on the test set
y_pred_survival = estimator.predict(X=X_test_survival)


# Evaluate the model
c_index = concordance_index_censored(y_test_survival['overall survival event'], y_test_survival['overall survival days'], y_pred_survival)
print(f"C-index: {c_index[0]}")

In [ ]:
pd.DataFrame(y_pred_survival)

In [ ]:

# save estimator
with open('estimator_survival_clinical.pkl', 'wb') as f:
    pickle.dump(estimator, f)
# Save the scaler
with open('scaler_survival_clinical.pkl', 'wb') as f:
    pickle.dump(scaler_selected, f)
# load the models
import pickle
estimator = pickle.load(open('estimator_survival_clinical.pkl', 'rb'))
scaler = pickle.load(open('scaler_survival_clinical.pkl', 'rb'))

print ("Calculating the survival Model")




In [ ]:
estimator = pickle.load(open('estimator_survival_clinical.pkl', 'rb'))
scaler = pickle.load(open('scaler_survival_clinical.pkl', 'rb'))

### Get best outcome based on clinical data

In [ ]:
def get_best_outcome_threatment(clinical_data:pd.DataFrame, days:int=365, estimator:CoxnetSurvivalAnalysis=estimator, scaler:StandardScaler=scaler) -> pd.DataFrame:
    """ This function takes a clinical data DataFrame and returns the best treatment outcome for each patient.
    Args:
        clinical_data (pd.DataFrame): DataFrame containing clinical data with features and survival information.
    Returns:
        daraframe: A dataframe with the treatments survival probabilities
    """
    # Ensure the clinical data is in the correct format
    data = clinical_data.copy()
    data = data[['age at diagnosis', 'tumor size', 'lymph node group',
       'lymph node status', 'er status', 'pgr status', 'her2 status',
       'ki67 status', 'nhg', 'pam50 subtype']]
    data_endocrine = data.copy()
    data_chemo = data.copy()
    data_both = data.copy()
    data_none = data.copy()
    data_endocrine['endocrine treated']= 1
    data_endocrine['chemo treated'] = 0
    data_chemo['endocrine treated'] = 0
    data_chemo['chemo treated'] = 1
    data_both['endocrine treated'] = 1
    data_both['chemo treated'] = 1
    data_none['endocrine treated'] = 0
    data_none['chemo treated'] = 0

    # Calculate cumulative hazard functions for each treatment
    chf_endocrine = estimator.predict_cumulative_hazard_function(data_endocrine)
    chf_chemo = estimator.predict_cumulative_hazard_function(data_chemo)
    chf_both = estimator.predict_cumulative_hazard_function(data_both)
    chf_none = estimator.predict_cumulative_hazard_function(data_none)
    # Calculate survival probabilities at the specified time point
    survival_probabilities = {
        "endocrine": [np.exp(-chf(days)) for chf in chf_endocrine],
        "chemo": [np.exp(-chf(days)) for chf in chf_chemo],
        "both": [np.exp(-chf(days)) for chf in chf_both],
        "none": [np.exp(-chf(days)) for chf in chf_none],
    }
    
    # convert to DataFrame
    survival_probabilities = pd.DataFrame(survival_probabilities)
    # Add the best treatment outcome for each patient
    survival_probabilities['best_treatment'] = survival_probabilities.idxmax(axis=1)

    return survival_probabilities


In [ ]:
clinical_data_df_cat_no_nan = clinical_data_df_cat.dropna()
clinical_data_df_cat_no_nan[clinical_vars]

In [ ]:
clinical_values = clinical_data_df_cat_no_nan.iloc[0:10, :]
results = get_best_outcome_threatment(clinical_data=clinical_values)



In [ ]:
clinical_values[clinical_vars]

In [ ]:
results

## Usage

In [ ]:
#Example output to try

def predict_treatments_and_survival(one_person : pd.DataFrame, days:int=365):
  one_person.drop(columns=['overall survival days', 'overall survival event','treatment'], inplace=True, errors='ignore')  # Drop survival columns for feature matrix
  best_outcome = get_best_outcome_threatment(one_person, days=365)
  return best_outcome

import pickle
# save the encoder
with open('multi_output_model_selected.pkl', 'wb') as f:
    pickle.dump(multi_output_model_selected, f)
with open('multi_output_model_encoded.pkl', 'wb') as f:
    pickle.dump(multi_target_model_encoded, f)
# Save the scaler
with open('scaler_selected.pkl', 'wb') as f:
    pickle.dump(scaler_selected, f)
print ("The ClinicalModel and the Scaler are ready for it.")
# save the models
one_person = expression_matrix_df.iloc[0:2,:]  # Select one person

predict_treatments_and_survival(one_person)